# Organize a Video Library

Discover what's in a video collection, find the best organization strategy, then categorize everything into a complete library catalog.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import json
import os

from twelvelabs import TwelveLabs, TextParam
from twelvelabs.types.text_param_format import TextParamFormat_JsonSchema

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "<YOUR_API_KEY>")
STORE_ID = os.environ.get("TWELVELABS_STORE_ID", "<YOUR_KNOWLEDGE_STORE_ID>")  # Replace with your knowledge store ID

client = TwelveLabs(api_key=API_KEY)

## Helper Functions

Utility functions for parsing Jockey API responses and printing results.

In [ ]:
def parse_response(response) -> str:
    """Extract text content from a Jockey response.

    Args:
        response: The ResponseObject returned by client.responses.create().

    Returns:
        The text content from the first message output, or an empty string
        if no message content is found.
    """
    for output in response.output:
        if output.type == "message":
            for content in output.content:
                return content.text
    return ""


def parse_json_response(response) -> dict:
    """Extract and parse JSON content from a Jockey response."""
    text = parse_response(response)
    if text:
        return json.loads(text)
    return {}

## Step 1: Get a Collection Overview

Start by asking Jockey for a high-level overview of what's in your video collection. This establishes context for the session and helps identify themes, subjects, and patterns across videos.

Best results come from collections with **10+ videos** for meaningful organization.

In [ ]:
print("Step 1: Understanding your collection...")

response = client.responses.create(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "Give me a high-level overview of this video collection. "
                "What themes, subjects, and patterns do you see?"
            ),
        }
    ],
)

session_id = response.session_id

overview_text = parse_response(response)
print(overview_text)

## Step 2: Discover Organization Axes

Ask Jockey to recommend the best ways to organize the collection. Using structured output with a JSON schema, the response returns ranked organization axes with expected categories and scores.

### Axes Schema

The schema below captures recommended organization strategies, each with a name, rationale, expected category list, and a relevance score.

In [ ]:
AXES_SCHEMA = {
    "type": "object",
    "properties": {
        "recommended_axes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "axis": {"type": "string"},
                    "reason": {"type": "string"},
                    "expected_categories": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "score": {"type": "number"},
                },
            },
        },
        "best_axis": {"type": "string"},
        "reasoning": {"type": "string"},
    },
}

In [ ]:
print("Step 2: Finding best organization strategy...")

response = client.responses.create(
    knowledge_store_id=STORE_ID,
    session_id=session_id,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "What are the best ways to organize this collection? "
                "Rank the top 3 axes by how well they'd separate "
                "the content into useful groups."
            ),
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="organization_axes", schema_=AXES_SCHEMA)
    ),
)

axes = parse_json_response(response)

print(f"Best axis: {axes['best_axis']}")
print(f"Reasoning: {axes['reasoning']}")
for ax in axes["recommended_axes"]:
    print(f"  {ax['axis']} (score: {ax['score']}): {ax['reason']}")

## Step 3: Organize by the Best Axis

Now apply the top-ranked organization axis to categorize every video. The catalog schema captures category names, descriptions, and a list of videos in each category.

### Catalog Schema

The schema below defines the complete library catalog output, organized into categories with video references, titles, and summaries.

In [ ]:
CATALOG_SCHEMA = {
    "type": "object",
    "properties": {
        "organization_axis": {"type": "string"},
        "categories": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "description": {"type": "string"},
                    "videos": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "reference": {"type": "string"},
                                "title": {"type": "string"},
                                "summary": {"type": "string"},
                            },
                        },
                    },
                },
            },
        },
        "total_videos": {"type": "integer"},
    },
}

In [ ]:
best_axis = axes["best_axis"]
print(f"Step 3: Organizing by '{best_axis}'...")

response = client.responses.create(
    knowledge_store_id=STORE_ID,
    session_id=session_id,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": f"Now organize every video by '{best_axis}'. Include all videos.",
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="video_catalog", schema_=CATALOG_SCHEMA)
    ),
)

catalog = parse_json_response(response)

print(f"\nLibrary Catalog ({catalog['total_videos']} videos)")
print(f"Organized by: {catalog['organization_axis']}")
for cat in catalog["categories"]:
    print(f"\n  [{cat['name']}] — {cat['description']}")
    for v in cat["videos"]:
        print(f"    {v['title']}: {v['summary']}")

## Shortcut: Organize by Known Criteria

If you already know how you want to organize your library, skip the discovery steps and classify directly. This is useful when you have predefined categories like content type, audience level, or topic.

### Direct Organization Schema

This schema includes an `uncategorized` field to catch any videos that don't fit neatly into your predefined categories.

In [ ]:
DIRECT_ORG_SCHEMA = {
    "type": "object",
    "properties": {
        "criteria": {"type": "string"},
        "categories": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "description": {"type": "string"},
                    "video_count": {"type": "integer"},
                    "videos": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "reference": {"type": "string"},
                                "reason": {"type": "string"},
                            },
                        },
                    },
                },
            },
        },
        "uncategorized": {"type": "array", "items": {"type": "string"}},
    },
}

In [ ]:
# Example: Organize by content type with predefined categories
response = client.responses.create(
    knowledge_store_id=STORE_ID,
    instructions=(
        "You are a content librarian. Organize videos into clear, "
        "mutually exclusive categories. Every video should appear "
        "in exactly one category."
    ),
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "Organize these videos by content type: tutorials, "
                "interviews, product demos, and announcements."
            ),
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="video_organization", schema_=DIRECT_ORG_SCHEMA)
    ),
)

org_result = parse_json_response(response)

for cat in org_result.get("categories", []):
    print(f"[{cat['name']}] ({cat['video_count']} videos): {cat['description']}")

if org_result.get("uncategorized"):
    print(f"\nUncategorized: {org_result['uncategorized']}")

## Example Criteria

You can swap the query in the shortcut above to organize by different criteria:

| Criteria | Query |
|----------|-------|
| By topic | "Organize by main subject matter" |
| By mood | "Organize by emotional tone: upbeat, serious, neutral" |
| By audience | "Organize by target audience: beginners, intermediate, advanced" |
| By format | "Organize by format: tutorial, interview, demo, vlog" |
| By speaker | "Group videos by who appears on screen" |

## Variations

- **Multi-axis catalog:** Run Step 3 multiple times with different axes from Step 2.
- **Hierarchical:** After the first organization, do a follow-up: "Now sub-organize the [category] group by difficulty level."
- **Export:** Pipe the JSON output to a file for use in your CMS or DAM system.

## Next Steps

- [Assemble Highlight Reels](./assemble_highlight_reels.ipynb) -- Find and assemble clips from your organized library.
- [Build a Content Agent](./build_content_agent.ipynb) -- Create a multi-step agent that produces structured creative output.
- [Organize Video Library (docs)](https://docs.twelvelabs.io/v1.3/agents/recipes/organize-a-video-library) -- Full reference documentation.